# 다이캐스팅 품질 예측 프로젝트 (Product Type 2 버전)

1. 데이터를 `product_type_2.csv` 기준으로 불러옵니다.

## 목표
1. 데이터를 불러오고(컬럼 구조 이해)
2. Defects를 **표면 / 구조 그룹**으로 다시 묶어 **Multi-label 타겟(y)** 을 만듭니다.
3. 공정(Process) + 센서(Sensor) 변수로 **XGBoost 모델**을 학습합니다.
4. **데이터 누출(leakage)** 위험이 있는 컬럼을 자동 제거합니다.
5. Feature importance / threshold tuning / “불량 최소 조건” 탐색까지 이어갑니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# XGBoost (설치가 안 되어 있으면 아래 주석을 해제 후 실행)
# !pip -q install xgboost
from xgboost import XGBClassifier

# SHAP for model interpretation
import shap

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

# 경고 무시 (선택사항)
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로드

이 프로젝트에서는 **`product_type_2.csv`** 를 사용합니다.

이 CSV는 **2줄 헤더(MultiIndex)** 구조일 수 있습니다.
- 상위 레벨: process / sensor / defects / defect_flag
- 하위 레벨: velocity_1, bubble_1, is_defect 등

그래서 `header=[0,1]`로 먼저 읽어보고,
- MultiIndex면 그대로 사용
- 아니면 일반 컬럼(flat)으로 처리합니다.


In [ ]:
DATA_PATH = "product_type_2.csv"  # 업로드된 Product Type 2 파일

try:
    df1 = pd.read_csv(DATA_PATH, header=[0, 1])
    multi = True
except Exception:
    df1 = pd.read_csv(DATA_PATH)
    multi = False

print("MultiIndex header:", multi)
print("Shape:", df1.shape)

if multi:
    print("Top-level columns:", df1.columns.levels[0].tolist())
else:
    print("Columns sample:", df1.columns[:15].tolist())

df1.head()


## 2. process / sensor / defects 블록 분리

- MultiIndex라면 `df1['process']` 처럼 접근 가능
- flat 컬럼이라면 `process_...`, `sensor_...`, `defects_...` 접두어로 찾아 분리

아래 코드는 두 경우를 모두 안전하게 처리합니다.


In [ ]:
def is_multiindex_columns(df: pd.DataFrame) -> bool:
    return isinstance(df.columns, pd.MultiIndex)

def get_block(df: pd.DataFrame, block_name: str) -> pd.DataFrame:
    if is_multiindex_columns(df):
        return df[block_name].copy()
    prefix = block_name.lower() + "_"
    cols = [c for c in df.columns if str(c).lower().startswith(prefix)]
    return df[cols].copy()

process_df = get_block(df1, "process")
sensor_df  = get_block(df1, "sensor")
defects_df = get_block(df1, "defects")

print("process_df:", process_df.shape)
print("sensor_df :", sensor_df.shape)
print("defects_df:", defects_df.shape)
print("\n[defects columns]")
print(defects_df.columns.tolist())


## 3. defect 그룹화 → multi-label 타겟(y) 만들기

- **표면(surface)**:
  - stain, dent, scratch, buring_mark, contamination, exfoliation
  - 겉면 외관 품질과 직접 연결되는 불량들

- **구조(structure)**:
  - short_shot, bubble, blow_hole, deformation, crack, impurity, inclusions
  - 내부 결함, 충진 문제, 구조 안정성과 더 가까운 불량들

주의:
- defects 값이 0/1 뿐 아니라 2,3도 있을 수 있으므로
  - **0이면 양품**
  - **0초과면 불량(1로 통일)**
  로 이진화합니다.
- 그룹 정의 후에는
  1. 각 defect 발생 수
  2. 그룹별 포함 여부
  3. 빠진 defect가 없는지
  를 반드시 점검합니다.


In [ ]:
defects_numeric = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_bin = (defects_numeric > 0).astype(int)

# - 표면(surface): 외관/표면 상태 관련
# - 구조(structure): 충진/내부/형상 안정성 관련
defect_groups = {
    "표면": [
        "stain_1", "stain_2",
        "dent_1", "dent_2",
        "scratch_1", "scratch_2",
        "buring_mark_1", "buring_mark_2",
        "contamination_1", "contamination_2",
        "exfoliation_1", "exfoliation_2",
    ],
    "구조": [
        "short_shot_1", "short_shot_2",
        "bubble_1", "bubble_2",
        "blow_hole_1", "blow_hole_2",
        "deformation_1", "deformation_2",
        "crack_1", "crack_2",
        "impurity_1", "impurity_2",
        "inclusions_1", "inclusions_2",
    ],
}

def normalize_defect_colname(col):
    s = str(col).lower()
    return s.replace("defects_", "").replace(" ", "")

# 실제 defects 컬럼명 표준화
defect_cols_actual = [normalize_defect_colname(c) for c in defects_bin.columns]
rename_map = {old: normalize_defect_colname(old) for old in defects_bin.columns}
defects_bin = defects_bin.rename(columns=rename_map)

# 실제 존재하는 컬럼만 그룹에 반영
group_cols_used = {}
for g, cols in defect_groups.items():
    usable = [c for c in cols if c in defects_bin.columns]
    group_cols_used[g] = usable

print("[그룹별 사용 컬럼]")
for g, cols in group_cols_used.items():
    print(f"- {g}: {len(cols)}개")
    print(" ", cols)

# 빠진 defect가 없는지 확인
all_grouped = sorted(set(sum(group_cols_used.values(), [])))
all_actual = sorted(defects_bin.columns.tolist())
ungrouped = [c for c in all_actual if c not in all_grouped]

print("\n[전체 defect 컬럼 수]", len(all_actual))
print("[그룹에 포함된 defect 컬럼 수]", len(all_grouped))
print("[그룹에서 빠진 defect 컬럼 수]", len(ungrouped))
if len(ungrouped) > 0:
    print("그룹에 포함되지 않은 defect:", ungrouped)
else:
    print("모든 defect 컬럼이 그룹에 포함되었습니다.")

# 개별 defect 발생 수 확인
defect_counts = defects_bin.sum().sort_values(ascending=False)
print("\n[개별 defect 발생 수]")
display(defect_counts.to_frame("count"))

# 그룹 타겟 생성
y = pd.DataFrame(index=defects_bin.index)
for g, cols in group_cols_used.items():
    if len(cols) == 0:
        y[g] = 0
    else:
        y[g] = defects_bin[cols].max(axis=1)

print("\n[y 분포]")
for col in y.columns:
    print(f"\n[{col}]")
    display(y[col].value_counts(dropna=False).sort_index().to_frame("count"))

print("\n[표면/구조 동시 발생 분포]")
combo = y.astype(str).agg("".join, axis=1)
display(combo.value_counts(dropna=False).sort_index().to_frame("count"))


## 4. X 만들기 + 데이터 누출(leakage) 방지

X는 **process + sensor**만 사용합니다.

그리고 아래는 “미래 예측에서 알 수 없거나”, “식별자/시간”이라서  
모델이 **공정 원인이 아니라 꼼수 패턴을 외우게** 만들 수 있습니다.

- process_id: 단순 식별번호
- process_shot: 생산 순서/카운터(시간 변수 성격)
- process_product_type: 이번 분석에서는 product_type_1만 사용 → 불필요/혼입 위험
- defect_flag_is_defect / is_defect: 이름부터 정답 힌트 가능

그래서 이런 컬럼을 **자동 탐지해서 제거**합니다.


In [ ]:
X = pd.concat([process_df, sensor_df], axis=1).copy()
X = X.drop(columns=["id","shot"], errors="ignore")
print("X shape (before drop):", X.shape)

def col_to_str(c):
    if isinstance(c, tuple):
        return "_".join([str(x) for x in c]).lower()
    return str(c).lower()

leak_keywords = [
    "process_id",
    "process_shot",
    "process_product_type",
    "defect_flag_is_defect",
    "is_defect",
]

leak_cols = [c for c in X.columns if any(k in col_to_str(c) for k in leak_keywords)]
if leak_cols:
    print("drop leakage cols:", leak_cols)
    X = X.drop(columns=leak_cols, errors="ignore")
else:
    print("leakage 키워드 컬럼 없음")

defect_like = [c for c in X.columns if "defects" in col_to_str(c)]
if defect_like:
    print("drop defects-like cols from X:", defect_like)
    X = X.drop(columns=defect_like, errors="ignore")

print("X shape (after drop):", X.shape)


In [ ]:
# leakage 제거 최종 확인
#아래 결과에 `shot`, `id`, `is_defect`, `defects` 관련 컬럼이 남아 있으면 안됨.

[c for c in X.columns if "defect" in str(c).lower() or "shot" in str(c).lower() or "id" in str(c).lower()]

## 5. Train/Test 분리 (multi-label stratify)

y가 (표면, 구조) 2개 컬럼이므로 `stratify=y`를 그대로 쓰면 오류가 나거나 불안정할 수 있습니다.

그래서 (표면,구조) 조합을 문자열로 만들어 strata로 사용합니다.
- 00, 10, 01, 11 같은 형태


In [ ]:
strata = y.astype(str).agg("".join, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

print("\n[라벨 조합 분포(전체)]")
display(strata.value_counts(normalize=True).sort_index().to_frame("ratio"))


## 6. XGBoost 모델 학습 (표면/구조 각각 1개 모델)

Multi-label이지만, 타겟(표면/구조)마다 불량 비율이 달라서  
타겟별로 `scale_pos_weight = (양품 수 / 불량 수)`를 따로 적용합니다.

평가 지표:
- ROC-AUC
- PR-AUC (불균형에서 특히 중요)
- precision / recall / f1


In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

models, preds, probas = {}, {}, {}

for t in y_train.columns:
    neg = int((y_train[t] == 0).sum())
    pos = int((y_train[t] == 1).sum())
    spw = float(neg / max(pos, 1))
    print(f"\n[{t}] scale_pos_weight = {spw:.3f} (neg={neg}, pos={pos})")

    model = XGBClassifier(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_lambda=1.0,
        scale_pos_weight=spw,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_imp, y_train[t])

    proba = model.predict_proba(X_test_imp)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    models[t] = model
    probas[t] = proba
    preds[t]  = pred

    print(classification_report(y_test[t], pred, digits=4))
    roc = roc_auc_score(y_test[t], proba)
    pr  = average_precision_score(y_test[t], proba)
    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")

y_pred  = pd.DataFrame(preds, index=y_test.index)
y_proba = pd.DataFrame(probas, index=y_test.index)

display(y_pred.head())
display(y_proba.head())


## 7. Threshold 튜닝

현재는 확률이 0.5 이상이면 불량이라고 판단합니다.

불량이 적은 데이터에서는 0.5가 너무 높아서 불량을 많이 놓칠 수 있으니,  
0.05~0.80 구간에서 threshold를 바꿔가며 precision/recall trade-off를 확인합니다.


In [ ]:
def evaluate_threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    rep = classification_report(y_true, pred, output_dict=True, zero_division=0)
    return {
        "threshold": threshold,
        "precision_1": rep["1"]["precision"],
        "recall_1": rep["1"]["recall"],
        "f1_1": rep["1"]["f1-score"],
        "accuracy": rep["accuracy"]
    }

for t in y_test.columns:
    rows = []
    for th in np.linspace(0.05, 0.80, 16):
        rows.append(evaluate_threshold(y_test[t].values, y_proba[t].values, float(th)))
    df_th = pd.DataFrame(rows)

    print(f"\n=== [{t}] threshold sweep (recall 높은 순 TOP5) ===")
    display(df_th.sort_values("recall_1", ascending=False).head(5))

    plt.figure(figsize=(6,4))
    plt.plot(df_th["threshold"], df_th["precision_1"], marker="o", label="precision(불량=1)")
    plt.plot(df_th["threshold"], df_th["recall_1"], marker="o", label="recall(불량=1)")
    plt.title(f"{t}: Precision/Recall vs Threshold")
    plt.xlabel("threshold")
    plt.ylabel("score")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.show()


< recall 기준 - threshold 낮을수록 불량 탐지 ↑ /
Accuracy 기준 - threshold 높을수록 증가 >
하지만 불균형 데이터에서는 accuracy 중요하지 않다고 판단, 즉 recall이 더 중요
결국 threshold = 0.10 으로 수정하기로 결정함.


Threshold 튜닝완료

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

models, preds, probas = {}, {}, {}

for t in y_train.columns:
    neg = int((y_train[t] == 0).sum())
    pos = int((y_train[t] == 1).sum())
    spw = float(neg / max(pos, 1))
    print(f"\n[{t}] scale_pos_weight = {spw:.3f} (neg={neg}, pos={pos})")

    model = XGBClassifier(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_lambda=1.0,
        scale_pos_weight=spw,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_imp, y_train[t])

    proba = model.predict_proba(X_test_imp)[:, 1]
    threshold = 0.1
    pred = (proba >= threshold).astype(int)

    models[t] = model
    probas[t] = proba
    preds[t]  = pred

    print(classification_report(y_test[t], pred, digits=4))
    roc = roc_auc_score(y_test[t], proba)
    pr  = average_precision_score(y_test[t], proba)
    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")

y_pred  = pd.DataFrame(preds, index=y_test.index)
y_proba = pd.DataFrame(probas, index=y_test.index)

display(y_pred.head())
display(y_proba.head())


## 8. Feature Importance

XGBoost는 각 변수의 중요도를 제공합니다.  
중요도가 높다고 “원인”이라고 단정할 수는 없지만,  
공정 점검/최적화 후보를 고르는 데 매우 유용합니다.


In [ ]:
for t, model in models.items():
    imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(f"\n=== [{t}] TOP 15 Feature Importance ===")
    display(imp.head(15).to_frame("importance"))

    plt.figure(figsize=(8,5))
    topk = imp.head(15)[::-1]
    plt.barh(topk.index.astype(str), topk.values)
    plt.title(f"{t} - Top 15 Feature Importances (XGBoost)")
    plt.xlabel("importance")
    plt.show()


## 9. “불량 최소 조건” 탐색

중요도 TOP 변수들을 10분위(Decile)로 나눠 구간별 불량률을 계산합니다.

- 어떤 구간에서 불량률이 가장 낮은지 확인
- 공정 조건 최적화 후보를 좁히는 데 도움

In [ ]:
TOP_N = 5

for t, model in models.items():
    print(f"\n=== [{t}] 불량 최소 조건 탐색 (TOP {TOP_N}) ===")
    imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    top_vars = imp.head(TOP_N).index.tolist()

    df_tmp = X_test.copy()
    df_tmp["y_true"] = y_test[t].values

    rows = []
    for v in top_vars:
        try:
            bins = pd.qcut(df_tmp[v], q=10, duplicates="drop")
            rate = df_tmp.groupby(bins)["y_true"].mean()
            rows.append({
                "var": str(v),
                "best_bin": str(rate.idxmin()),
                "best_defect_rate": float(rate.min())
            })
        except Exception as e:
            rows.append({
                "var": str(v),
                "best_bin": "N/A",
                "best_defect_rate": np.nan,
                "note": repr(e)
            })

    display(pd.DataFrame(rows))


## 10. SHAP 분석을 통한 모델 해석

**SHAP (SHapley Additive exPlanations)**는 게임 이론의 Shapley value를 기반으로 머신러닝 모델의 예측을 설명합니다.

### SHAP의 장점
1. **개별 예측 설명** - "왜 이 제품이 불량으로 예측되었는가?"
2. **방향성 제공** - 각 변수가 불량 확률을 증가/감소시키는 방향 파악
3. **전역 + 국소 해석** - 전체 패턴과 개별 케이스 모두 분석 가능
4. **수학적 엄밀성** - Shapley value는 유일한 공정 분배 방법

### 분석 순서
1. SHAP Explainer 생성
2. 전역 중요도 분석 (Summary Plot)
3. 주요 변수 심층 분석 (Dependence Plot)
4. 개별 불량품 원인 분석 (Waterfall Plot)
5. 표면 vs 구조 불량 원인 비교
6. 양품 vs 불량 공정 조건 비교
7. 실무 인사이트 도출

In [ ]:
# ========================================
# SHAP Explainer 생성
# ========================================
print("SHAP Explainer 생성 중...")

# 모델이 실제로 학습/예측에 사용한 입력과 동일한 형태로 SHAP용 데이터 구성
X_train_imp_df = pd.DataFrame(X_train_imp, columns=X_train.columns, index=X_train.index)
X_test_imp_df  = pd.DataFrame(X_test_imp,  columns=X_train.columns, index=X_test.index)

def _to_2d_shap_array(shap_values):
    """shap 버전에 따라 list / ndarray / Explanation으로 달라질 수 있어 2차원 배열로 정리"""
    if hasattr(shap_values, "values"):
        shap_values = shap_values.values
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            shap_values = shap_values[1]
        else:
            shap_values = shap_values[0]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]
    return shap_values

# 타깃별 SHAP 결과 저장
shap_results = {}

for t in models.keys():
    print(f"{t} 불량 SHAP values 계산 중...")
    explainer = shap.TreeExplainer(models[t])
    raw_shap_values = explainer.shap_values(X_test_imp_df)
    shap_values = _to_2d_shap_array(raw_shap_values)

    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)):
        expected_value = np.asarray(expected_value).reshape(-1)[-1]

    shap_results[t] = {
        "explainer": explainer,
        "shap_values": shap_values,
        "expected_value": float(expected_value)
    }

# 기존 셀들과의 호환성을 위해 별도 변수도 유지
explainer_surface = shap_results["표면"]["explainer"]
explainer_structure = shap_results["구조"]["explainer"]
shap_values_surface = shap_results["표면"]["shap_values"]
shap_values_structure = shap_results["구조"]["shap_values"]

print(f"\n표면 SHAP shape: {shap_values_surface.shape}")
print(f"구조 SHAP shape: {shap_values_structure.shape}")
print("SHAP 계산 완료! ✓")


### 10.1 전역 중요도 분석 (Summary Plot)

**Summary Plot**은 모든 샘플에 대한 SHAP values의 분포를 보여줍니다.

- **세로축**: 특성 중요도 순위
- **가로축**: SHAP value (양수 = 불량 확률 증가, 음수 = 감소)
- **색상**: 특성 값 (빨강 = 높음, 파랑 = 낮음)
- **점의 분포**: 영향도의 다양성

예: 빨간 점이 오른쪽에 모여있으면 → "높은 값이 불량 확률을 증가시킴"

In [ ]:
# ========================================
# Summary Plot - Bar (Feature Importance)
# ========================================
print("=== 표면 불량 - Feature Importance (SHAP) ===")
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_surface, 
    X_test_imp, 
    feature_names=X.columns.tolist(),
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title("표면 불량 - SHAP Feature Importance (절대값 평균)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\n=== 구조 불량 - Feature Importance (SHAP) ===")
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_structure, 
    X_test_imp, 
    feature_names=X.columns.tolist(),
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title("구조 불량 - SHAP Feature Importance (절대값 평균)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Summary Plot - Beeswarm (상세 분포)
# ========================================
print("=== 표면 불량 - SHAP Summary (Beeswarm) ===")
plt.figure(figsize=(10, 10))
shap.summary_plot(
    shap_values_surface, 
    X_test_imp, 
    feature_names=X.columns.tolist(),
    max_display=20,
    show=False
)
plt.title("표면 불량 - SHAP Summary Plot (Beeswarm)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\n=== 구조 불량 - SHAP Summary (Beeswarm) ===")
plt.figure(figsize=(10, 10))
shap.summary_plot(
    shap_values_structure, 
    X_test_imp, 
    feature_names=X.columns.tolist(),
    max_display=20,
    show=False
)
plt.title("구조 불량 - SHAP Summary Plot (Beeswarm)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

### 10.2 주요 변수 심층 분석 (Dependence Plot)

**Dependence Plot**은 특정 변수의 값에 따라 SHAP value가 어떻게 변하는지 보여줍니다.

- **가로축**: 특성 값
- **세로축**: SHAP value (불량 확률에 미치는 영향)
- **색상**: 자동으로 선택된 상호작용 변수

이를 통해:
1. **비선형 관계** 파악 (예: "온도 720°C 이상에서 급격히 불량 증가")
2. **임계값** 발견 (예: "압력 260 이하로 유지해야 함")
3. **상호작용** 확인 (예: "온도가 높을 때 압력의 영향이 더 큼")

In [ ]:
# ========================================
# Dependence Plot - 표면 불량 주요 변수
# ========================================
# 상위 3개 중요 변수 선택
shap_importance_surface = pd.Series(
    np.abs(shap_values_surface).mean(axis=0),
    index=X.columns
).sort_values(ascending=False)

top_features_surface = shap_importance_surface.head(3).index

print("=== 표면 불량 - 주요 변수 Dependence Plot ===")
print(f"분석 대상: {list(top_features_surface)}\n")

for feat in top_features_surface:
    feat_idx = X.columns.get_loc(feat)
    
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(
        feat_idx, 
        shap_values_surface, 
        X_test_imp,
        feature_names=X.columns.tolist(),
        interaction_index="auto",
        show=False
    )
    plt.title(f"표면 불량 - {feat}의 영향도 분석", fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# ========================================
# Dependence Plot - 구조 불량 주요 변수
# ========================================
shap_importance_structure = pd.Series(
    np.abs(shap_values_structure).mean(axis=0),
    index=X.columns
).sort_values(ascending=False)

top_features_structure = shap_importance_structure.head(3).index

print("=== 구조 불량 - 주요 변수 Dependence Plot ===")
print(f"분석 대상: {list(top_features_structure)}\n")

for feat in top_features_structure:
    feat_idx = X.columns.get_loc(feat)
    
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(
        feat_idx, 
        shap_values_structure, 
        X_test_imp,
        feature_names=X.columns.tolist(),
        interaction_index="auto",
        show=False
    )
    plt.title(f"구조 불량 - {feat}의 영향도 분석", fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
    print()

### 10.3 개별 불량품 원인 분석 (Waterfall Plot)

**Waterfall Plot**은 하나의 예측을 기본 확률에서 시작하여 각 변수가 순차적으로 확률을 어떻게 변화시키는지 보여줍니다.

읽는 법:
```
E[f(X)] = 0.08 (기본 불량률 8%)
+ 온도 높음: +0.25
+ 압력 낮음: +0.18
- 속도 적정: -0.08
= f(x) = 0.43 (최종 예측: 43%)
```

이를 통해 **"왜 이 제품이 불량으로 예측되었는가?"**를 명확히 설명할 수 있습니다.

In [ ]:
# ========================================
# Waterfall Plot - 표면 불량 예시
# ========================================
# 불량으로 예측된 샘플 중 확률이 높은 것 선택
defect_indices_surface = np.where(y_test["표면"] == 1)[0]

if len(defect_indices_surface) > 0:
    # 예측 확률이 가장 높은 샘플 선택
    proba_defects = probas["표면"][defect_indices_surface]
    top_defect_idx = defect_indices_surface[np.argmax(proba_defects)]
    
    print(f"=== 표면 불량 예측 분석 (샘플 #{top_defect_idx}) ===")
    print(f"실제 라벨: {y_test.iloc[top_defect_idx]['표면']}")
    print(f"예측 확률: {probas['표면'][top_defect_idx]:.4f}")
    print(f"기본 확률: {explainer_surface.expected_value:.4f}\n")
    
    # SHAP Explanation 객체 생성
    explanation = shap.Explanation(
        values=shap_values_surface[top_defect_idx],
        base_values=explainer_surface.expected_value,
        data=X_test_imp[top_defect_idx],
        feature_names=X.columns.tolist()
    )
    
    # Waterfall plot
    plt.figure(figsize=(10, 8))
    shap.waterfall_plot(explanation, max_display=15, show=False)
    plt.title(f"표면 불량 - 원인 분해 (샘플 #{top_defect_idx})", fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("테스트 데이터에 표면 불량 샘플이 없습니다.")

In [ ]:
# ========================================
# Waterfall Plot - 구조 불량 예시
# ========================================
defect_indices_structure = np.where(y_test["구조"] == 1)[0]

if len(defect_indices_structure) > 0:
    # 예측 확률이 가장 높은 샘플 선택
    proba_defects = probas["구조"][defect_indices_structure]
    top_defect_idx = defect_indices_structure[np.argmax(proba_defects)]
    
    print(f"=== 구조 불량 예측 분석 (샘플 #{top_defect_idx}) ===")
    print(f"실제 라벨: {y_test.iloc[top_defect_idx]['구조']}")
    print(f"예측 확률: {probas['구조'][top_defect_idx]:.4f}")
    print(f"기본 확률: {explainer_structure.expected_value:.4f}\n")
    
    explanation = shap.Explanation(
        values=shap_values_structure[top_defect_idx],
        base_values=explainer_structure.expected_value,
        data=X_test_imp[top_defect_idx],
        feature_names=X.columns.tolist()
    )
    
    plt.figure(figsize=(10, 8))
    shap.waterfall_plot(explanation, max_display=15, show=False)
    plt.title(f"구조 불량 - 원인 분해 (샘플 #{top_defect_idx})", fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("테스트 데이터에 구조 불량 샘플이 없습니다.")

### 10.4 표면 불량 vs 구조 불량 원인 비교

두 가지 불량 유형의 주요 원인을 비교하여:
1. **공통 원인** 파악 (두 불량 모두에 영향)
2. **차별적 원인** 발견 (한 쪽에만 크게 영향)

이를 통해 **타겟별 맞춤 공정 관리 전략**을 수립할 수 있습니다.

In [ ]:
# ========================================
# 표면 vs 구조 불량 원인 비교
# ========================================
comparison_targets = pd.DataFrame({
    "표면_중요도": shap_importance_surface,
    "구조_중요도": shap_importance_structure
})

comparison_targets["차이"] = (
    comparison_targets["표면_중요도"] - comparison_targets["구조_중요도"]
)

print("=" * 70)
print("표면 불량 vs 구조 불량 - 주요 원인 비교")
print("=" * 70)

print("\n[표면 불량에 더 중요한 변수 TOP 5]:")
surface_specific = comparison_targets.sort_values("차이", ascending=False).head(5)
print(surface_specific.round(4))

print("\n[구조 불량에 더 중요한 변수 TOP 5]:")
structure_specific = comparison_targets.sort_values("차이", ascending=True).head(5)
print(structure_specific.round(4))

print("\n[양쪽 모두 중요한 공통 변수 TOP 5]:")
comparison_targets["평균_중요도"] = (
    comparison_targets[["표면_중요도", "구조_중요도"]].mean(axis=1)
)
comparison_targets["차이_절대값"] = comparison_targets["차이"].abs()
common_important = comparison_targets.sort_values(
    ["평균_중요도", "차이_절대값"], 
    ascending=[False, True]
).head(5)
print(common_important[["표면_중요도", "구조_중요도", "평균_중요도"]].round(4))

In [ ]:
# ========================================
# 시각화: 표면 vs 구조 중요도 비교
# ========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_n = 15

# 표면 불량
shap_importance_surface.head(top_n)[::-1].plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title("표면 불량 - SHAP Feature Importance", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Mean |SHAP value|")
axes[0].grid(axis='x', alpha=0.3)

# 구조 불량
shap_importance_structure.head(top_n)[::-1].plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title("구조 불량 - SHAP Feature Importance", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Mean |SHAP value|")
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### 10.5 양품 vs 불량 공정 조건 비교

SHAP 분석 결과를 바탕으로 **실제 공정 데이터를 비교**하여:
1. 불량품에서 어떤 변수가 비정상적인 값을 보이는지
2. 양품의 공정 조건은 어떤 범위에 있는지
3. 구체적인 개선 목표 수치 도출

**목표**: "온도를 X°C로 낮추면 불량률 Y% 감소" 같은 구체적 제안

In [ ]:
# ========================================
# 표면 불량: 양품 vs 불량 조건 비교
# ========================================
defect_mask_surface = y_test["표면"] == 1
good_mask_surface = y_test["표면"] == 0

# 불량품의 평균 SHAP values (불량을 증가시키는 변수)
defect_shap_surface = shap_values_surface[defect_mask_surface]
mean_shap_defect_surface = pd.Series(
    defect_shap_surface.mean(axis=0),
    index=X.columns
).sort_values(ascending=False)

print("=" * 70)
print("표면 불량: 양품 vs 불량 공정 조건 비교")
print("=" * 70)

print("\n[불량품에서 평균적으로 불량을 증가시킨 TOP 10 변수]:")
print(mean_shap_defect_surface.head(10).round(4))

# 공정 조건 비교 테이블
# FIX: y_test의 인덱스와 맞추기 위해 index 지정
X_test_df = pd.DataFrame(X_test_imp, columns=X.columns, index=y_test.index)

comparison_surface = pd.DataFrame({
    "양품_평균": X_test_df[good_mask_surface].mean(),
    "불량_평균": X_test_df[defect_mask_surface].mean(),
    "차이": X_test_df[defect_mask_surface].mean() - X_test_df[good_mask_surface].mean(),
    "SHAP_기여도": mean_shap_defect_surface
})

comparison_surface["차이율(%)"] = (
    comparison_surface["차이"] / comparison_surface["양품_평균"].replace(0, 1) * 100
)

# SHAP 기여도가 큰 변수 중 실제 차이도 큰 것 찾기
critical_features_surface = comparison_surface.sort_values(
    "SHAP_기여도", ascending=False
).head(10)

print("\n[양품 vs 불량 공정 조건 비교 - SHAP 기여도 TOP 10]:")
print(critical_features_surface.round(4))

In [ ]:
# ========================================
# 구조 불량: 양품 vs 불량 조건 비교
# ========================================
defect_mask_structure = y_test["구조"] == 1
good_mask_structure = y_test["구조"] == 0

defect_shap_structure = shap_values_structure[defect_mask_structure]
mean_shap_defect_structure = pd.Series(
    defect_shap_structure.mean(axis=0),
    index=X.columns
).sort_values(ascending=False)

print("=" * 70)
print("구조 불량: 양품 vs 불량 공정 조건 비교")
print("=" * 70)

print("\n[불량품에서 평균적으로 불량을 증가시킨 TOP 10 변수]:")
print(mean_shap_defect_structure.head(10).round(4))

# X_test_df는 이미 위에서 생성되어 있음 (y_test.index 사용)
comparison_structure = pd.DataFrame({
    "양품_평균": X_test_df[good_mask_structure].mean(),
    "불량_평균": X_test_df[defect_mask_structure].mean(),
    "차이": X_test_df[defect_mask_structure].mean() - X_test_df[good_mask_structure].mean(),
    "SHAP_기여도": mean_shap_defect_structure
})

comparison_structure["차이율(%)"] = (
    comparison_structure["차이"] / comparison_structure["양품_평균"].replace(0, 1) * 100
)

critical_features_structure = comparison_structure.sort_values(
    "SHAP_기여도", ascending=False
).head(10)

print("\n[양품 vs 불량 공정 조건 비교 - SHAP 기여도 TOP 10]:")
print(critical_features_structure.round(4))

### 10.6 실무 인사이트 및 공정 개선 제안

SHAP 분석 결과를 종합하여 **실행 가능한 공정 개선 제안**을 도출합니다.

제안 형식:
- **변수명**: [현재 불량품 평균] → [권장 범위(양품 기준)]
- **예상 효과**: 불량률 X% 감소
- **우선순위**: SHAP 기여도 기반

In [ ]:
# ========================================
# 공정 개선 제안 - 표면 불량
# ========================================
print("=" * 70)
print("공정 개선 제안 - 표면 불량")
print("=" * 70)

for i, (feat, row) in enumerate(critical_features_surface.head(5).iterrows(), 1):
    good_val = row["양품_평균"]
    bad_val = row["불량_평균"]
    diff = row["차이"]
    diff_pct = row["차이율(%)"]
    shap_contrib = row["SHAP_기여도"]
    
    direction = "높음" if diff > 0 else "낮음"
    recommendation = "낮추기" if diff > 0 else "높이기"
    
    # 양품의 범위 (5th ~ 95th percentile)
    # FIX: .loc 대신 boolean indexing 직접 사용
    good_data = X_test_df[good_mask_surface][feat]
    good_min = good_data.quantile(0.05)
    good_max = good_data.quantile(0.95)
    
    print(f"\n[우선순위 #{i}] {feat}")
    print(f"  • SHAP 기여도: {shap_contrib:+.4f} (불량 확률 증가)")
    print(f"  • 양품 평균: {good_val:.4f}")
    print(f"  • 불량 평균: {bad_val:.4f}")
    print(f"  • 차이: {diff:+.4f} ({diff_pct:+.2f}%)")
    print(f"  • 불량품에서 {direction}")
    print(f"  • 권장 조치: {recommendation}")
    print(f"  • 권장 범위: {good_min:.4f} ~ {good_max:.4f} (양품 5~95 백분위수)")

In [ ]:
# ========================================
# 공정 개선 제안 - 구조 불량
# ========================================
print("=" * 70)
print("공정 개선 제안 - 구조 불량")
print("=" * 70)

for i, (feat, row) in enumerate(critical_features_structure.head(5).iterrows(), 1):
    good_val = row["양품_평균"]
    bad_val = row["불량_평균"]
    diff = row["차이"]
    diff_pct = row["차이율(%)"]
    shap_contrib = row["SHAP_기여도"]
    
    direction = "높음" if diff > 0 else "낮음"
    recommendation = "낮추기" if diff > 0 else "높이기"
    
    # 양품의 범위 (5th ~ 95th percentile)
    # FIX: .loc 대신 boolean indexing 직접 사용
    good_data = X_test_df[good_mask_structure][feat]
    good_min = good_data.quantile(0.05)
    good_max = good_data.quantile(0.95)
    
    print(f"\n[우선순위 #{i}] {feat}")
    print(f"  • SHAP 기여도: {shap_contrib:+.4f} (불량 확률 증가)")
    print(f"  • 양품 평균: {good_val:.4f}")
    print(f"  • 불량 평균: {bad_val:.4f}")
    print(f"  • 차이: {diff:+.4f} ({diff_pct:+.2f}%)")
    print(f"  • 불량품에서 {direction}")
    print(f"  • 권장 조치: {recommendation}")
    print(f"  • 권장 범위: {good_min:.4f} ~ {good_max:.4f} (양품 5~95 백분위수)")

In [ ]:
# ========================================
# 최종 요약 및 결론
# ========================================
print("\n" + "=" * 70)
print("SHAP 분석 최종 요약")
print("=" * 70)

print("\n주요 발견사항:")
print("\n1. 표면 불량 주요 원인:")
for feat in shap_importance_surface.head(3).index:
    print(f"   - {feat}: {shap_importance_surface[feat]:.4f}")

print("\n2. 구조 불량 주요 원인:")
for feat in shap_importance_structure.head(3).index:
    print(f"   - {feat}: {shap_importance_structure[feat]:.4f}")

print("\n3. 차별적 원인:")
print(f"   - 표면 불량에 특히 중요: {comparison_targets.sort_values('차이', ascending=False).index[0]}")
print(f"   - 구조 불량에 특히 중요: {comparison_targets.sort_values('차이', ascending=True).index[0]}")

print("\n4. 공통 중요 변수:")
for feat in common_important.index[:3]:
    print(f"   - {feat}: 표면 {comparison_targets.loc[feat, '표면_중요도']:.4f}, "
          f"구조 {comparison_targets.loc[feat, '구조_중요도']:.4f}")

print("\n권장 조치:")
print("   1. 상위 5개 변수의 공정 조건을 양품 범위로 조정")
print("   2. 표면/구조 불량별로 차별화된 관리 전략 수립")
print("   3. 실시간 모니터링으로 이상 징후 조기 감지")
print("   4. SHAP 기반 알람 시스템 구축 (예: 불량 확률 70% 이상 시 경고)")

print("\n" + "=" * 70)
print("SHAP 분석 완료!")
print("=" * 70)

## SHAP 기반 주요 공정 변수 요약

`shap_results`에 저장된 타깃별 SHAP 값을 바탕으로  
각 불량 유형(표면 / 구조)에서 중요하게 작용한 상위 변수를 정리한다.

이때 SHAP 계산은 **모델이 실제로 입력받은 전처리 후 데이터(`X_test_imp_df`) 기준**으로 수행한다.


In [ ]:
import numpy as np
import pandas as pd

# 타깃별 SHAP 중요도 표 생성
shap_importance_dict = {}

for t in models.keys():
    shap_values = shap_results[t]["shap_values"]

    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    shap_importance_dict[t] = importance_df

    print(f"===== [{t}] SHAP 중요 변수 TOP 10 =====")
    display(importance_df.head(10))


## 양품 vs 불량 공정 변수 분포 비교

각 타깃별 SHAP 상위 변수 3개를 선택하여  
전처리 후 테스트 데이터(`X_test_imp_df`) 기준으로 양품과 불량의 분포 차이를 비교한다.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 타깃별로 SHAP 상위 3개 변수의 양품/불량 분포 비교
for t in y_test.columns:
    print(f"===== [{t}] 양품 vs 불량 공정 변수 분포 비교 =====")
    top_features = shap_importance_dict[t].head(3)["feature"].tolist()

    df_box = X_test_imp_df.copy()
    df_box["target"] = y_test[t].values

    for col in top_features:
        plt.figure(figsize=(6, 4))
        sns.boxplot(data=df_box, x="target", y=col)
        plt.title(f"[{t}] Defect vs {col}")
        plt.xlabel("Defect (0=Good, 1=Defect)")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()


## 공정 변수 구간별 불량률 분석

각 타깃별 SHAP 최상위 변수 1개를 기준으로 값을 5구간으로 나누고,  
구간별 불량률을 계산하여 위험 구간과 상대적 안정 구간을 탐색한다.


In [ ]:
# 타깃별 상위 중요 변수 1개에 대해 구간별 불량률 분석
for t in y_test.columns:
    df_analysis = X_test_imp_df.copy()
    df_analysis["defect"] = y_test[t].values

    target_var = shap_importance_dict[t].iloc[0]["feature"]

    try:
        df_analysis["bin"] = pd.qcut(df_analysis[target_var], 5, duplicates="drop")
        bin_defect_rate = df_analysis.groupby("bin", observed=False)["defect"].mean().reset_index()

        print(f"===== [{t}] {target_var} 구간별 불량률 =====")
        display(bin_defect_rate)

        plt.figure(figsize=(8, 4))
        plt.plot(bin_defect_rate["bin"].astype(str), bin_defect_rate["defect"], marker="o")
        plt.xticks(rotation=45)
        plt.title(f"[{t}] {target_var} 구간별 불량률")
        plt.xlabel("구간")
        plt.ylabel("Defect Rate")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"[{t}] {target_var} 구간화 실패: {e}")


## 공정 해석 및 시사점

SHAP 분석과 공정 변수 분포 분석 결과 일부 공정 변수들이 불량 발생에 큰 영향을 미치는 것으로 나타났다.

특정 변수 값이 일정 범위를 초과하거나 감소할 경우 불량 발생 확률이 증가하는 경향이 확인되었다.
이러한 결과는 공정 관리 기준 설정 및 실시간 품질 예측 시스템 구축에 활용될 수 있다.


## 10.7 SHAP dependence plot 기반 공정 임계값 분석

상위 중요 변수에 대해 값의 구간별 불량률과 SHAP 영향 방향을 함께 확인하여,  
어느 범위에서 불량 위험이 높아지는지 탐색한다.

- SHAP 중요도 상위 5개 변수 선정
- 변수별 5분위 구간으로 나누어 불량률 계산
- SHAP 방향성과 실제 불량률을 함께 고려하여 위험 구간 해석
- 공정 관리 후보 범위 제안


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SHAP 입력용 데이터프레임 안전하게 재구성
if "X_test_imp_df" not in globals():
    try:
        X_test_imp_df = pd.DataFrame(X_test_imp, columns=X_train.columns, index=X_test.index)
    except Exception:
        X_test_imp_df = X_test.copy()

def _get_shap_array(shap_values):
    # shap 버전에 따라 list 또는 ndarray일 수 있음
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            return np.array(shap_values[1])
        return np.array(shap_values[0])
    return np.array(shap_values)

def _interval_to_text(interval_obj):
    try:
        left = float(interval_obj.left)
        right = float(interval_obj.right)
        return f"{left:.3f} ~ {right:.3f}"
    except Exception:
        return str(interval_obj)

def _direction_text(feature_series, shap_feature):
    # 변수값 증가 시 SHAP 값(불량 방향 영향)이 증가하는지 확인
    corr = pd.Series(feature_series).corr(pd.Series(shap_feature), method="spearman")
    if pd.isna(corr):
        return "변동 방향 해석이 불안정함"
    if corr >= 0.15:
        return "값이 커질수록 불량 방향 영향이 커지는 경향"
    elif corr <= -0.15:
        return "값이 작을수록 불량 방향 영향이 커지는 경향"
    else:
        return "값의 크기와 불량 영향 방향이 뚜렷하지 않음"

threshold_results = {}

for t in models.keys():
    shap_arr = _get_shap_array(shap_results[t]["shap_values"])
    importance_df = pd.DataFrame({
        "feature": X_test_imp_df.columns,
        "mean_abs_shap": np.abs(shap_arr).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    top5_features = importance_df.head(5)["feature"].tolist()
    print(f"\n{'='*80}")
    print(f"[{t}] SHAP 상위 5개 변수 기반 공정 임계값 분석")
    print(f"{'='*80}")
    print(top5_features)

    rows = []

    for feature in top5_features:
        tmp = pd.DataFrame({
            "feature_value": X_test_imp_df[feature],
            "target": y_test[t].values
        }).dropna()

        # 값이 적어 qcut 실패할 수 있으므로 예외 처리
        try:
            tmp["bin"] = pd.qcut(tmp["feature_value"], q=5, duplicates="drop")
        except Exception:
            print(f"[주의] {feature}: 구간화 실패")
            continue

        summary = (
            tmp.groupby("bin", observed=False)
               .agg(sample_count=("target", "size"),
                    defect_rate=("target", "mean"),
                    value_min=("feature_value", "min"),
                    value_max=("feature_value", "max"))
               .reset_index()
        )

        if len(summary) == 0:
            continue

        # 위험 구간 = 불량률이 가장 높은 구간
        risk_row = summary.loc[summary["defect_rate"].idxmax()]
        safe_row = summary.loc[summary["defect_rate"].idxmin()]

        feature_idx = list(X_test_imp_df.columns).index(feature)
        shap_direction = _direction_text(X_test_imp_df[feature], shap_arr[:, feature_idx])

        rows.append({
            "feature": feature,
            "risk_interval": f"{risk_row['value_min']:.3f} ~ {risk_row['value_max']:.3f}",
            "risk_defect_rate": float(risk_row["defect_rate"]),
            "safe_interval": f"{safe_row['value_min']:.3f} ~ {safe_row['value_max']:.3f}",
            "safe_defect_rate": float(safe_row["defect_rate"]),
            "direction": shap_direction
        })

        print(f"\n[변수] {feature}")
        display(summary)
        print(f"→ 위험 구간(불량률 최대): {risk_row['value_min']:.3f} ~ {risk_row['value_max']:.3f} | 불량률={risk_row['defect_rate']:.3f}")
        print(f"→ 안정 구간(불량률 최소): {safe_row['value_min']:.3f} ~ {safe_row['value_max']:.3f} | 불량률={safe_row['defect_rate']:.3f}")
        print(f"→ SHAP 방향 해석: {shap_direction}")

        # 시각화
        plt.figure(figsize=(7, 4))
        x_labels = [f"{row.value_min:.2f}~{row.value_max:.2f}" for row in summary.itertuples()]
        plt.plot(x_labels, summary["defect_rate"], marker="o")
        plt.title(f"[{t}] {feature} 구간별 불량률")
        plt.xlabel(feature)
        plt.ylabel("Defect Rate")
        plt.xticks(rotation=25)
        plt.grid(alpha=0.3)
        plt.show()

    threshold_results[t] = pd.DataFrame(rows)

print("\n분석 완료: threshold_results 딕셔너리에 타깃별 임계값 후보가 저장되었습니다.")


## 10.8 핵심 공정 변수 TOP 5 및 자동 해석 문장

아래 셀은 SHAP 중요도와 구간별 불량률 분석 결과를 바탕으로  
각 타깃별 핵심 공정 변수 TOP 5와 해석 문장을 자동으로 생성한다.

이 문장은 프로젝트 보고서, 발표 자료, 노트북 결론 부분에 그대로 참고할 수 있다.


In [ ]:

def _feature_label(feature_name):
    return (feature_name
            .replace("process_", "Process: ")
            .replace("sensor_", "Sensor: ")
            .replace("_", " ")
            .title())

for t, result_df in threshold_results.items():
    print(f"\n{'#'*90}")
    print(f"[{t}] 핵심 공정 변수 TOP 5 및 해석 문장")
    print(f"{'#'*90}")

    if result_df.empty:
        print("해석 가능한 결과가 없습니다.")
        continue

    display(result_df)

    print("\n[보고서용 해석 문장]")
    for i, row in result_df.reset_index(drop=True).iterrows():
        feat = _feature_label(row["feature"])
        risk_interval = row["risk_interval"]
        risk_rate = row["risk_defect_rate"]
        safe_interval = row["safe_interval"]
        safe_rate = row["safe_defect_rate"]
        direction = row["direction"]

        sentence = (
            f"{i+1}. {feat}는(은) 불량 예측에 중요한 변수로 나타났다. "
            f"특히 {risk_interval} 구간에서 불량률이 {risk_rate:.1%}로 가장 높았고, "
            f"{safe_interval} 구간에서는 {safe_rate:.1%}로 상대적으로 낮았다. "
            f"SHAP 분석 기준으로는 '{direction}'로 해석되므로, "
            f"해당 변수는 공정 관리 우선순위 변수로 볼 수 있다."
        )
        print(sentence)


## 10.9 공정 관리 범위 제안 및 최종 프로젝트 결론

이 단계에서는 모델 성능, SHAP 분석, 구간별 불량률 분석을 종합하여  
실제 제조 현장에서 관리가 필요한 공정 변수와 우선 관리 범위를 정리한다.

핵심은 단순히 '예측 정확도'를 보여주는 데서 끝나는 것이 아니라,  
'어떤 공정 조건에서 불량이 증가하는가'와  
'어떤 범위를 우선 관리해야 하는가'를 제안하는 것이다.


In [ ]:

for t, result_df in threshold_results.items():
    print(f"\n{'='*100}")
    print(f"[{t}] 공정 관리 범위 제안")
    print(f"{'='*100}")

    if result_df.empty:
        print("제안할 결과가 없습니다.")
        continue

    top3 = result_df.sort_values("risk_defect_rate", ascending=False).head(3).reset_index(drop=True)

    print("[우선 관리 권장 변수 TOP 3]")
    for i, row in top3.iterrows():
        print(
            f"- {row['feature']}: 위험 구간 {row['risk_interval']} "
            f"(불량률 {row['risk_defect_rate']:.1%}) / "
            f"상대적 안정 구간 {row['safe_interval']} "
            f"(불량률 {row['safe_defect_rate']:.1%})"
        )

    print("\n[최종 결론 문장 예시]")
    print(
        f"{t} 불량 예측 모델은 주요 공정 변수의 변화가 불량 확률에 미치는 영향을 효과적으로 포착하였다. "
        f"특히 상위 중요 변수 일부는 특정 구간에서 불량률이 뚜렷하게 상승하는 패턴을 보였으며, "
        f"이는 해당 구간을 공정 관리 기준 후보로 활용할 수 있음을 시사한다. "
        f"따라서 본 프로젝트는 단순 불량 예측을 넘어, 공정 조건 해석과 관리 범위 제안까지 가능한 제조 데이터 분석 프로젝트로 확장될 수 있다."
    )
